# Data Cleaning

In [2]:
import pandas as pd

# Read in files
df_corn = pd.read_csv("../data/Corn-Table.csv", header = None)
df_cotton = pd.read_csv("../data/Cotton-Table.csv", header = None)
df_soybeans = pd.read_csv("../data/Soybeans-Table.csv", header = None)

All of the three data sets are not in the correct format currently. The data needs to be reformatted into a data frame. Each of them is taken apart and put into columns using the function below.

In [136]:
# Create a parse function

def parse(df):   
    results = []
    current_attr = None

    for i, row in df.iterrows():
        first_cell = str(row.iloc[0]).strip()

        # Find attributes
        if "percent of all" in first_cell:
            current_attr = first_cell.strip()
            continue

        # Find State/Year
        if first_cell == "State/Year":
            years = row[1:].tolist()
            continue

        # States and associated values
        state = first_cell
        values = row.iloc[1:len(years)+1]

        # Skip empty rows
        if first_cell == "nan" or first_cell.strip() == "":
            continue

        # Create sub data frame
        temp = pd.DataFrame({
            "State": state,
            "Year": years,
            "Value": values
        })

        temp["Attribute"] = current_attr
        results.append(temp)

    # Combine all data frames
    df_long = pd.concat(results, ignore_index=True)

    # Clean values
    df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce")

    return df_long

A side effect of the messy data is that some empty rows were not caught, those are removed below.

In [137]:
corn = parse(df_corn)
corn = corn.dropna(subset=["Year"])
cotton = parse(df_cotton)
cotton = cotton.dropna(subset=["Year"])
soybeans = parse(df_soybeans)
soybeans = soybeans.dropna(subset=["Year"])

In order to keep all of the crops separate and labeled, a crop column is added to the data set.

In [138]:
corn["Crop"] = "Corn"
cotton["Crop"] = "Cotton"
soybeans["Crop"] = "Soybeans"

All of the data sets now share all of the same columns, so they can be stacked vertically easily.

In [139]:
df_all = pd.concat([corn, cotton, soybeans], ignore_index=True)

Next I want to check all data types and fix them as necessary.

In [140]:
print(df_all.dtypes)

State         object
Year         float64
Value        float64
Attribute     object
Crop          object
dtype: object


In [141]:
df_all["Year"] = df_all["Year"].astype(int)

Some of the state names didn't parse properly so they were fixed.

In [142]:
df_all["State"] = df_all["State"].str.replace("North Dakota 2/", "North Dakota")
df_all["State"] = df_all["State"].str.replace("Texas 2/", "Texas")
df_all["State"] = df_all["State"].str.replace("Other States 1/", "Other States")
df_all["State"] = df_all["State"].str.replace("Alabama 2/", "Alabama")
df_all["State"] = df_all["State"].str.replace("Missouri 2/", "Missouri")
df_all["State"] = df_all["State"].str.replace("Tennessee 2/", "Tennessee")

In [143]:
print(df_all["State"].unique())

['Illinois' 'Indiana' 'Iowa' 'Kansas' 'Michigan' 'Minnesota' 'Missouri'
 'Nebraska' 'North Dakota' 'Ohio' 'South Dakota' 'Texas' 'Wisconsin'
 'Other States' 'United States' 'Alabama' 'Arkansas' 'California'
 'Georgia' 'Louisiana' 'Mississippi' 'North Carolina' 'Tennessee']


In [144]:
df_all.to_csv('../data/new_data.csv', index=False)

In [3]:
farm_income = pd.read_csv("../data/farm_income.csv")

In [4]:
farm_income.dtypes

Rank                  object
State                 object
Farm_income          float64
Share_income         float64
Cumulative_income    float64
dtype: object